GPT-4'ün regex desenini ekle. Metni önce GPT-4'ün deseniyle parçalara böl, BPE'yi her parçada ayrı çalıştır. Tokenizer'ı yeniden eğit, birleştirmeleri öncekiyle karşılaştır. Sonra "Ankara'dan" ve "İstanbul'da" gibi kesme işaretli kelimeleri bu desenle böl. Desen İngilizce için yazılmış: Türkçede ne yapıyor, kendi cümlelerinle yaz. (0:57:36 - 1:11:38)

In [ ]:
import regex as re
import pandas as pd

### GPT-4 deseni

In [ ]:
gpt4pat = re.compile(r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+""")

In [11]:
ornek = "Hello've world123 how's are you!!!?    Merhaba dünya, 2024 yılında 12345 kişi geldi!!"
print(gpt4pat.findall(ornek))

['Hello', "'ve", ' world', '123', ' how', "'s", ' are', ' you', '!!!?', '   ', ' Merhaba', ' dünya', ',', ' ', '202', '4', ' yılında', ' ', '123', '45', ' kişi', ' geldi', '!!']


In [4]:
with open('felatun_bey_tam_metin.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [5]:
text_chunks = gpt4pat.findall(text)
print("parça sayısı:", len(text_chunks))
print(text_chunks[2000:2040])

parça sayısı: 75726
[',', ' hiç', ' olmazsa', ' \n', '65', ' kazanç', ',', ' gelir', ';', ' söyleme', ';', ' getirme', '\n', '66', ' filozofça', ' değerlendirmeler', '\n', '67', ' ince', '\n', 'Anadolu', ' Üniversitesi', '\n', '8', '\n', 'fazl', '68', ' u', ' kemâlini', ' ', '69', ' ise', ' kendisi', ' beğenmiş', ' olduğundan', ' cuma', ' günü', ' mutlaka', ' \n', 'bir']


In [6]:
def get_stats(ids, counts=None):
    counts = {} if counts is None else counts
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

In [ ]:
def merge(ids, pair, idx):
    newids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i += 1
    return newids

In [8]:
vocab_size = 1000
num_merges = vocab_size - 256

def build_vocab(merges):
    vocab = {idx: bytes([idx]) for idx in range(256)}
    for (p0, p1), idx in merges.items():
        vocab[idx] = vocab[p0] + vocab[p1]
    return vocab

def show(b):
    return b.decode("utf-8", errors="replace")

### regexsiz tüm metin tek parça

In [9]:
tokens = list(text.encode("utf-8"))
ids = list(tokens)
merges_basic = {}
for i in range(num_merges):
    stats = get_stats(ids)
    pair = max(stats, key=stats.get)
    idx = 256 + i
    ids = merge(ids, pair, idx)
    merges_basic[pair] = idx
vocab_basic = build_vocab(merges_basic)
n_basic = len(ids)
print(f"basic -> token: {n_basic}, sıkıştırma: {len(tokens) / n_basic:.2f}X")

basic -> token: 154399, sıkıştırma: 2.60X


### GPT-4 regexli Tokenizer


In [ ]:
ids_chunks = [list(ch.encode("utf-8")) for ch in text_chunks] # UTF-8 bytelarına çevir
merges_regex = {}
for i in range(num_merges):
    stats = {}
    for chunk_ids in ids_chunks:
        get_stats(chunk_ids, stats)
    pair = max(stats, key=stats.get)
    idx = 256 + i
    ids_chunks = [merge(chunk_ids, pair, idx) for chunk_ids in ids_chunks]
    merges_regex[pair] = idx
vocab_regex = build_vocab(merges_regex)
n_regex = sum(len(c) for c in ids_chunks)
print(f"regex -> token: {n_regex}, sıkıştırma: {len(tokens) / n_regex:.2f}X")

regex -> token: 153307, sıkıştırma: 2.61X


### Birleştirmeleri karşılaştır

In [12]:
def merge_list(merges, vocab, n):
    return [f"{show(vocab[p0])!r} + {show(vocab[p1])!r} -> {show(vocab[idx])!r}"
            for (p0, p1), idx in list(merges.items())[:n]]

pd.set_option("display.max_rows", 100)
pd.DataFrame({
    "basic": merge_list(merges_basic, vocab_basic, 30),
    "gpt4 regex": merge_list(merges_regex, vocab_regex, 30),
}, index=range(256, 286))

,basic,gpt4 regex
256,'�' + '�' -> 'ı','�' + '�' -> 'ı'
257,'n' + ' ' -> 'n ',' ' + 'b' -> ' b'
258,'e' + ' ' -> 'e ','�' + '�' -> 'â'
259,'�' + '�' -> 'â','e' + 'r' -> 'er'
260,'i' + ' ' -> 'i ','�' + '�' -> 'ş'
261,'e' + 'r' -> 'er','a' + 'n' -> 'an'
262,'�' + '�' -> 'ş','i' + 'n' -> 'in'
263,'a' + ' ' -> 'a ','�' + '�' -> 'ü'
264,'�' + '�' -> 'ü','a' + 'r' -> 'ar'
265,'a' + 'r' -> 'ar','e' + 'n' -> 'en'


In [13]:
set_basic = set(vocab_basic[i] for i in range(256, vocab_size))
set_regex = set(vocab_regex[i] for i in range(256, vocab_size))
print("ortak token sayısı      :", len(set_basic & set_regex))
print("sadece basic'te olanlar :", len(set_basic - set_regex))
print("sadece regex'te olanlar :", len(set_regex - set_basic))

ortak token sayısı      : 290
sadece basic'te olanlar : 454
sadece regex'te olanlar : 454


### Regex tokenizer ile encode / decode

In [15]:
def encode_chunk(text_bytes, merges):
    ids = list(text_bytes)
    while len(ids) >= 2:
        stats = get_stats(ids)
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break
        ids = merge(ids, pair, merges[pair])
    return ids

def encode_regex(text):
    ids = []
    for chunk in gpt4pat.findall(text):
        ids.extend(encode_chunk(chunk.encode("utf-8"), merges_regex))
    return ids

def decode_regex(ids):
    return b"".join(vocab_regex[i] for i in ids).decode("utf-8", errors="replace")

s = "Felâtun Bey ile Râkım Efendi İstanbul'da buluştular 😊"
ids = encode_regex(s)
print([show(vocab_regex[i]) for i in ids])
print(decode_regex(ids) == s)

['Felâtun', ' Bey', ' ile', ' Râkım', ' Efendi', ' İ', 'st', 'an', 'b', 'ul', "'", 'd', 'a', ' bu', 'lu', 'ş', 't', 'u', 'lar', ' ', '�', '�', '�', '�']
True


### Kesme işaretli kelimeler

In [16]:
kelimeler = ["Ankara'dan", "İstanbul'da", "Türkiye'nin", "Ayşe'ye", "Kars'ta", "Ankara'ma",
             "Paris'se", "Kadıköy'deki", "Ali'nin", "Mehmet'le", "ANKARA'DAN", "2024'te"]
for k in kelimeler:
    print(f"{k:15} -> {gpt4pat.findall(k)}")

Ankara'dan      -> ['Ankara', "'d", 'an']
İstanbul'da     -> ['İstanbul', "'d", 'a']
Türkiye'nin     -> ['Türkiye', "'nin"]
Ayşe'ye         -> ['Ayşe', "'ye"]
Kars'ta         -> ['Kars', "'t", 'a']
Ankara'ma       -> ['Ankara', "'m", 'a']
Paris'se        -> ['Paris', "'s", 'e']
Kadıköy'deki    -> ['Kadıköy', "'d", 'eki']
Ali'nin         -> ['Ali', "'nin"]
Mehmet'le       -> ['Mehmet', "'le"]
ANKARA'DAN      -> ['ANKARA', "'D", 'AN']
2024'te         -> ['202', '4', "'t", 'e']


In [17]:
for k in ["it's", "don't", "we're", "I've", "she'll", "you'd", "I'm"]:
    print(f"{k:8} -> {gpt4pat.findall(k)}")

it's     -> ['it', "'s"]
don't    -> ['don', "'t"]
we're    -> ['we', "'re"]
I've     -> ['I', "'ve"]
she'll   -> ['she', "'ll"]
you'd    -> ['you', "'d"]
I'm      -> ['I', "'m"]


In [19]:
cumle = "Ankara'dan İstanbul'a giderken Kadıköy'deki arkadaşım Ayşe'ye uğradım, 2024'te tekrar geleceğim!"
print(gpt4pat.findall(cumle))
print([show(vocab_regex[i]) for i in encode_regex(cumle)])

['Ankara', "'d", 'an', ' İstanbul', "'a", ' giderken', ' Kadıköy', "'d", 'eki', ' arkadaşım', ' Ayşe', "'ye", ' uğradım', ',', ' ', '202', '4', "'t", 'e', ' tekrar', ' geleceğim', '!']
['An', 'k', 'ar', 'a', "'", 'd', 'an', ' İ', 'st', 'an', 'b', 'ul', "'", 'a', ' gi', 'der', 'k', 'en', ' K', 'ad', 'ık', 'ö', 'y', "'", 'd', 'ek', 'i', ' ar', 'k', 'ad', 'aş', 'ım', ' A', 'y', 'ş', 'e', "'", 'ye', ' u', 'ğr', 'ad', 'ım', ',', ' ', '20', '2', '4', "'", 't', 'e', ' tek', 'r', 'ar', ' ge', 'le', 'ceğ', 'im', '!']
